# Playground S6E8 (Predicting Smartphone Addiction) / `Why Every S6E8 Notebook Above 0.97110 Overfits`

- **コンペ**: [Predicting Smartphone Addiction — Playground Series S6E8](https://www.kaggle.com/competitions/playground-series-s6e8)（残り12日）
- **原著notebook**: [Why Every S6E8 Notebook Above 0.97110 Overfits](https://www.kaggle.com/code/raykkretzschmar/why-every-s6e8-notebook-above-0-97110-overfits)
- **原著者**: RAYK KRETZSCHMAR (raykkretzschmar) ・ 56 votes（Silver）・ Apache 2.0 ・ Version 22 / ランタイム 1分7秒
- **スコア**: Public 0.97115（**現在の公開LB最上位帯**。ただし著者自身が「これは選ばない」と明言）

> ⚠️ これは**学習目的の解説付き写し**です。コード本体は原著のまま変更していませんが、実行結果（outputs）は含みません。
> コードの著作権は原著者に帰属します（Apache 2.0）。

## なぜこのnotebookを選んだか

このVaultではS6E8をすでに16本扱っており、その大半が「アンサンブル／スタッキングでLBを0.0000X上げる」内容でした。
17本目に同じことをするより、**「その0.0000Xの上げ幅に意味はあるのか」を実データで検証したnotebook**のほうが学習価値が高いと判断しました。

このnotebookが特異なのは、**公開LB 0.97115（現在の最上位帯）を出しておきながら、著者自身が
「この提出は私のprivate選択には入れない」と書いている**点です。自分の高スコアを自分で解剖する、という構成になっています。

## 手法の概要 — 3つの検証

| # | 検証 | 主張 |
|---|---|---|
| 1 | OOFとLBの**符号が逆** | 正直なOOFでは「生徒モデルを+12%足すと良い」、公開LBでは「-8%引くと良い」。真逆の答えが出る |
| 2 | **疑似公開LBシミュレーション** | OOF内に「59,260行の偽の公開LB」を10回作り、そこで最良の重みを選ぶと、選択に使わなかった行では負ける |
| 3 | **Season 6 の実績backtest** | 完了済み7エピソードのうち**3回は公開top10の生存者がゼロ**。S6E7では公開1位がprivate 440位 |

## 評価指標

- **タスク**: テーブルデータからスマートフォン依存（`addicted_label`）を予測する**2値分類**。
- **指標**: **ROC-AUC**。予測値の「順位」だけで決まり、絶対値のキャリブレーションは問われません。
  だからこのnotebookも最後は確率ではなく**パーセンタイル順位**を提出しています。
- **なぜこの指標か**: 陽性が少ない不均衡データで「全員陰性」と答えても高得点になってしまう accuracy と違い、
  AUCは**陽性と陰性のペアを正しく順序づけられた割合**なので、不均衡に対して頑健です。
- **★ この指標の「怖さ」がこのnotebookの主題**: 公開LBは全テスト296,302行のうち約20%（≈59,260行）だけで計算されます。
  AUC 0.97 付近では、**5桁目（0.00001）の差は数十件のペアの順序が入れ替わっただけ**で生じます。
  つまり `0.97115 > 0.97110` は「モデルが良い」ではなく「**59,260行というサンプルで、たまたまそうなった**」の可能性が高い。
  このnotebookはその可能性を、推測ではなく**実測**で示しています。

## この手法が指標に対してどう設計されているか

著者は「LBを上げる」ためではなく「**LBの上げ幅が本物かを判定する**」ために指標を使っています。
具体的には、同じ候補集合に対して (a) 全OOFでのAUC（真値に近い）、(b) 公開LBサイズのサブサンプルでのAUC（観測値）、
(c) 選択に使わなかった残りでのAUC（未来の代理）の**3つを比較**し、(b)で選ぶと(c)が下がることを示します。
これは統計学でいう**選択バイアス／勝者の呪い（winner's curse）**の教科書的な実験デザインです。


# Why every S6E8 notebook above 0.97110 probably overfits

## Including this one

The last cell produces a **0.97115 public-LB submission**. I would not select it for the private leaderboard.

That sounds contradictory, but it is the point of the notebook. Near the top of this competition, a fifth decimal can be manufactured by choosing among many almost-identical predictions after seeing public scores. The underlying models may be excellent; the *choice between their final micro-adjustments* is what overfits.

Three checks are run below:

1. an honest OOF signal points in one direction while the public leaderboard rewards the opposite direction;
2. repeated pseudo-public selection creates a gain on the selected split and a loss on the unused split;
3. the completed Season 6 leaderboards show how often public top tens disappeared on the private board.

All inputs are public: [Naji's OOF/submission library](https://www.kaggle.com/datasets/najiama/predicting-smartphone-addiction-oof-submission-csv), the [teacher/student signals](https://www.kaggle.com/datasets/raykkretzschmar/s6e8-transductive-anti-student-signals), and [Georgy Mamarin's public/private Season 6 boards](https://www.kaggle.com/datasets/georgymamarin/playground-series-s6-leaderboards).

### 【解説】セル1: データの読み込みと「順位化」の道具

**何をしているか**: 4つの公開データセット（Najiのteacher OOF・提出CSV、著者のstudent信号、Season 6の全LB）を読み込みます。
補助関数として `locate`（Kaggle上とローカルの両方でファイルを探す）、`pct_rank`、`strict_rank` を定義します。

**なぜそうするのか**:
- **`locate` の二段構え**: `/kaggle/input/**` を再帰的に探し、無ければ手元のディレクトリを遡ります。
  同じnotebookをKaggle上でもローカルでも動かすための実務的な工夫です。
- **`pct_rank`（パーセンタイル順位）**: 値そのものではなく「何番目か」を 0〜1 に直します。
  AUCは順位しか見ないので、**スケールの違うモデル同士を混ぜるときは順位に直してから混ぜる**のが定石です
  （確率のまま平均すると、自信過剰なモデルが平均を支配します）。
- **`strict_rank`（決定論的タイブレーク付き順位）**: `np.lexsort` で**同点のときの順序まで固定**します。
  同点をどう並べるかで5桁目のAUCは動きます。ここを乱数任せにすると「再実行したらスコアが変わる」ことになり、
  そもそも検証が成立しません。**測定したいものより測定誤差が大きい状況では、まず測定器を固定する**という発想です。

**用語**: *OOF (Out-Of-Fold)* = 交差検証で「その行を学習に使っていないモデル」が出した予測。
訓練データに対する正直な予測値で、CVスコアの計算元になります。
*teacher/student* = 大きな強いモデル（teacher）の出力を、小さなモデル（student）に学習させる「蒸留」の呼び方。


In [ ]:
import glob
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import rankdata, spearmanr
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedShuffleSplit

TARGET = "addicted_label"

def locate(kaggle_pattern, local_relative):
    matches = glob.glob(f"/kaggle/input/**/{kaggle_pattern}", recursive=True)
    if matches:
        return Path(matches[0])
    for root in [Path.cwd(), *Path.cwd().parents]:
        candidate = root / local_relative
        if candidate.exists():
            return candidate
    raise FileNotFoundError((kaggle_pattern, local_relative))

def pct_rank(x):
    x = np.asarray(x)
    return (rankdata(x, method="average") - 0.5) / len(x)

def strict_rank(x, tie_breaker):
    order = np.lexsort((np.asarray(tie_breaker), np.asarray(x)))
    out = np.empty(len(order), dtype=np.float64)
    out[order] = (np.arange(len(order)) + 0.5) / len(order)
    return out

train_path = locate("train.csv", "data/train.csv")
oof_path = locate("19_blend_oof_predictions.csv", "research_aug17/naji_latest/19_blend_oof_predictions.csv")
test_path = locate("19_blend_submission.csv.csv", "research_aug17/naji_latest/19_blend_submission.csv.csv")
signals_path = locate("transductive_signals.npz", "research_aug19/signal_dataset/transductive_signals.npz")
boards_path = locate("s6_leaderboards.csv", "research_aug19/history/s6_leaderboards.csv")

train = pd.read_csv(train_path)
naji_oof = pd.read_csv(oof_path).set_index("id").reindex(train["id"])
naji_test = pd.read_csv(test_path)
signals = np.load(signals_path)
y = train[TARGET].to_numpy()
teacher_oof = pct_rank(naji_oof[TARGET].to_numpy())
student_oof = pct_rank(signals["oof_soft_student"])

assert not naji_oof[TARGET].isna().any()
assert len(student_oof) == len(train)
print(f"train: {len(train):,} | test: {len(naji_test):,} | positive rate: {y.mean():.3f}")

## 1. The leaderboard rewards the wrong sign

The student is a seven-variable, five-fold OOF distillation model. It is weaker than the large public stack, but its residual is useful: adding about 12% improves the teacher on all 15 validation slices used during development.

Then I submitted the fixed weights. The leaderboard preferred **subtracting** the student—the exact opposite of the OOF result.

The table combines freshly computed OOF AUC with the public scores recorded for those exact files. Submission references are included so the result is auditable rather than reconstructed from memory.

### 【解説】セル3: 検証1 —「OOFとLBが正反対を指す」を表にする

**何をしているか**: 生徒モデルの重み `w` を -0.20 〜 +0.30 まで 0.01 刻みで変え、
`(1-w)*teacher + w*student` のOOF AUCを全部計算します。そこに**実際に提出して記録した公開LBスコア4件**を
突き合わせて表とグラフにします。

**なぜそうするのか**:
- **`submission_ref`（提出ID）が書かれている点に注目**。「記憶では上がった気がする」ではなく、
  **後から誰でも監査できる形**で証拠を残しています。分析notebookとしての誠実さの手本です。
- **結果の読み方**: OOFは「w=+0.12 あたりが最良」と言い、公開LBは「w=-0.08 が最良（0.97115）」と言います。
  **符号が逆**です。生徒モデルの残差が本当に有用なら、この2つは同じ方向を向くはずです。
- **なぜ「符号の逆転」がスコア差より強い証拠なのか**: 「OOFとLBの値が少しズレている」だけなら測定誤差で説明できます。
  しかし**最適な方向そのものが反転する**なら、公開LBの59,260行に固有の何か（サンプリングノイズ、
  公開/private間の分布差、公開行特有のクセ）を拾っている疑いが濃い。
  そしてそのどれであっても、**隠されたprivateラベルでも同じ反転が起きる保証はありません**。
- **著者の禁欲的な結論の書き方**: 「Najiのスタックが悪いとは言っていない。**最後の微調整を公開LBで選ぶことが検証ではない**、と言っている」。
  主張の範囲を必要以上に広げていません。批判的な分析ほど、この線引きが重要です。


In [ ]:
weights = np.round(np.arange(-0.20, 0.301, 0.01), 2)
oof_curve = pd.DataFrame({
    "student_weight": weights,
    "oof_auc": [roc_auc_score(y, (1-w)*teacher_oof + w*student_oof) for w in weights],
})
base_auc = float(oof_curve.loc[oof_curve.student_weight.eq(0), "oof_auc"].iloc[0])
oof_curve["delta_vs_teacher"] = oof_curve.oof_auc - base_auc

public_ledger = pd.DataFrame([
    {"student_weight": -0.12, "public_lb": 0.97114, "submission_ref": 55584411},
    {"student_weight": -0.08, "public_lb": 0.97115, "submission_ref": 55584395},
    {"student_weight":  0.00, "public_lb": 0.97113, "submission_ref": "published Naji v19 artifact"},
    {"student_weight":  0.12, "public_lb": 0.97103, "submission_ref": 55584375},
])
audit = public_ledger.merge(oof_curve, on="student_weight")
display(audit[["student_weight", "oof_auc", "delta_vs_teacher", "public_lb", "submission_ref"]]
        .style.format({"oof_auc":"{:.8f}", "delta_vs_teacher":"{:+.8f}", "public_lb":"{:.5f}"}))

fig, ax = plt.subplots(figsize=(9, 4.6))
ax.plot(oof_curve.student_weight, oof_curve.delta_vs_teacher * 1e5, lw=2, label="OOF delta")
ax.axvline(0, color="0.6", lw=1)
ax.axhline(0, color="0.6", lw=1)
ax.scatter(audit.student_weight, audit.delta_vs_teacher * 1e5, s=60, zorder=3)
for _, r in audit.iterrows():
    ax.annotate(f"LB {r.public_lb:.5f}", (r.student_weight, r.delta_vs_teacher*1e5),
                xytext=(5, 7), textcoords="offset points", fontsize=9)
ax.set(xlabel="weight on the honest OOF student", ylabel="OOF AUC change (×1e-5)",
       title="OOF says add the student; public LB says subtract it")
ax.grid(alpha=.2)
plt.show()

best_oof = oof_curve.loc[oof_curve.oof_auc.idxmax()]
print(f"OOF optimum: w={best_oof.student_weight:+.2f}, delta={best_oof.delta_vs_teacher:+.8f}")
print("Public optimum among submitted weights: w=-0.08, despite an OOF loss of",
      f"{audit.loc[audit.student_weight.eq(-.08), 'delta_vs_teacher'].iloc[0]:+.8f}")

This is stronger evidence than “the scores are close.” The model signal transfers to OOF consistently, yet the public test slice asks for its mirror image. Possible explanations include sampling noise, a public/private distribution difference, or artifacts specific to the public rows. None justifies trusting the mirror correction on the hidden private labels.

It does **not** prove that Naji's stack, the student, or every high-scoring notebook is bad. It proves that selecting this last direction by public LB is not model validation.

## 2. A miniature leaderboard in OOF

The real public set contains about 20% of 296,302 test rows, approximately 59,260 labels. The following experiment repeatedly exposes exactly 59,260 OOF rows as a pseudo-public board, chooses the best weight, and evaluates that choice on all untouched rows.

This is deliberately a friendly search: only 11 weights for one legitimate signal. A real leaderboard search over notebooks, seeds, bands, blends and post-processing choices has a much larger multiple-testing burden.

### 【解説】セル6: 検証2 — OOFの中に「偽の公開リーダーボード」を作る

**何をしているか**: OOFデータから **59,260行（＝実際の公開LBと同じサイズ）** を層化抽出して「疑似公開LB」とし、
11個の候補の重みからそこで最良のものを選びます。そして**選択に使わなかった残りの行**でそのスコアを評価します。
これを10回繰り返します。

**なぜそうするのか（このnotebookで一番重要なセル）**:
- **`oracle_idx` を基準にしている**: 全OOFで最良の重み（＝ほぼ真の最適解）を「答え」とし、
  疑似公開LBで選んだ重みがそれとどれだけズレるかを測ります。
- **2つの数字の対比**が肝心です:
  - `selected_public_gain` = 選択に使った59,260行での見かけ上の伸び → **プラスになる**
  - `unseen_private_delta` = 使わなかった行での実際の差 → **平均でマイナスになる**
  つまり「**選んだ場所では上がって見えるが、選んでいない場所では下がっている**」。
  これが**選択バイアス（勝者の呪い）**そのものです。宝くじを100枚買って一番当たった1枚を見せられても、
  次も当たるとは限らないのと同じ理屈です。
- **`StratifiedShuffleSplit` を使う理由**: 陽性率を保ったまま分割するため。実際の公開/privateの分割も
  ランダムなので、この模擬が現実に近くなります。
- **★ 著者が「これは意図的に甘い実験だ」と断っている点**: 候補はたった11個、しかも「1つの筋の通った信号」だけ。
  現実のLB探索は**notebook・シード・帯域・ブレンド・後処理**の組み合わせで数百〜数千の候補を試します。
  候補が増えるほど「たまたま良く見えるもの」は増える（**多重検定の問題**）ので、
  **現実の過学習は、この実験が示す量よりずっと大きい**ことになります。
  弱い設定でも悪い結果が出るなら、強い設定ではもっと悪い —— という論法です。


In [ ]:
candidate_weights = np.round(np.arange(-0.20, 0.301, 0.05), 2)
candidate_pred = np.array([(1-w)*teacher_oof + w*student_oof for w in candidate_weights])
full_auc = np.array([roc_auc_score(y, p) for p in candidate_pred])
oracle_idx = int(full_auc.argmax())

trials = []
splitter = StratifiedShuffleSplit(n_splits=10, train_size=59_260, random_state=940813)
for trial, (public_idx, private_idx) in enumerate(splitter.split(np.zeros(len(y)), y), 1):
    public_auc = np.array([roc_auc_score(y[public_idx], p[public_idx]) for p in candidate_pred])
    chosen = int(public_auc.argmax())
    private_auc = np.array([roc_auc_score(y[private_idx], p[private_idx]) for p in candidate_pred])
    trials.append({
        "trial": trial,
        "chosen_weight": candidate_weights[chosen],
        "selected_public_gain": public_auc[chosen] - public_auc[oracle_idx],
        "unseen_private_delta": private_auc[chosen] - private_auc[oracle_idx],
    })

trials = pd.DataFrame(trials)
display(trials.style.format({"chosen_weight":"{:+.2f}", "selected_public_gain":"{:+.8f}",
                             "unseen_private_delta":"{:+.8f}"}))
print(f"Full-OOF oracle weight: {candidate_weights[oracle_idx]:+.2f}")
print(f"Mean apparent gain from public selection: {trials.selected_public_gain.mean():+.8f}")
print(f"Mean delta on labels not used for selection: {trials.unseen_private_delta.mean():+.8f}")
print(f"Different weight selected in {(trials.chosen_weight != candidate_weights[oracle_idx]).sum()}/10 trials")

## 3. Season 6 already ran this experiment for us

Seven Playground episodes have finished, so their public and private boards are an unusually relevant backtest. The table below asks a narrow question: if a team was public top 10, did it remain private top 10?

This does not isolate notebook users from private competitors, and episodes differ in synthetic-data construction. It does measure the risk attached to optimizing a very dense public frontier in the same competition series.

### 【解説】セル8: 検証3 — Season 6 の実績で「本当に起きたか」を確かめる

**何をしているか**: 完了済みのPlayground Season 6 全エピソードについて、
公開LB上位10チームのうち何チームがprivate上位10に残ったかを数え、順位相関（Spearman）と一緒に表示します。

**なぜそうするのか**:
- **シミュレーションだけでは「本当に起きること」の証拠にならない**ので、**実際に起きた過去**を確認しています。
  シミュレーション（検証2）→ 実データ（検証3）という順序は、主張を裏づける型として非常に良い流れです。
- **結果**: 7エピソード中**3回（S6E2, S6E6, S6E7）は公開top10の生存者がゼロ**。
  S6E7では**公開1位がprivate 440位**。同じシリーズの、同じ合成データ生成の枠組みで起きています。
- **「全体の順位相関は高いのに、なぜ先頭だけ壊れるのか」**: これがこのセルの一番の学びです。
  Spearman相関は全チームを見ているので、実力差の大きい中位〜下位が正しく並べば高くなります。
  一方、**先頭の10チームは互いに実力がほぼ同じ**で、順位を決めているのはノイズです。
  **「全体としては信頼できる指標」でも「先端では信頼できない」**——
  平均的な性能指標を見て安心してはいけない、という一般則です。
- `is_host_baseline` を除外している点も丁寧です。主催者のベースライン投稿を混ぜると順位が歪みます。


In [ ]:
boards = pd.read_csv(boards_path)
rows = []
for episode, g in boards.groupby("episode", sort=True):
    g = g.loc[~g.is_host_baseline].dropna(subset=["public_rank", "private_rank"])
    top10 = g.nsmallest(10, "public_rank")
    rows.append({
        "episode": episode,
        "teams": len(g),
        "all-rank correlation": spearmanr(g.public_rank, g.private_rank).statistic,
        "public top 10 still private top 10": int((top10.private_rank <= 10).sum()),
        "public winner private rank": int(g.nsmallest(1, "public_rank").private_rank.iloc[0]),
        "worst private rank in public top 10": int(top10.private_rank.max()),
    })
history = pd.DataFrame(rows)
display(history.style.format({"all-rank correlation":"{:.4f}"}))

fig, ax = plt.subplots(figsize=(9, 4.4))
colors = ["#cf3f3f" if v == 0 else "#3977b5" for v in history["public top 10 still private top 10"]]
ax.bar(history.episode, history["public top 10 still private top 10"], color=colors)
ax.set(ylim=(0, 10.5), ylabel="public top-10 teams still in private top 10",
       title="Three of seven Season 6 public top tens had zero survivors")
ax.axhline(10, color="0.5", ls="--", lw=1)
ax.grid(axis="y", alpha=.2)
plt.show()

High overall rank correlation is not protection at the frontier. S6E2, S6E6 and S6E7 each had **zero** public-top-10 teams remain in the private top 10. In S6E7—the most relevant previous episode—the public winner finished private rank 440.

That is why the threshold in the title is 0.97110. It is not a magical statistical boundary. It marks the region where improvements are mostly a few public-LB ten-thousandths, shared predictions are extremely correlated, and many choices have already been tried. Above it, the burden of proof should shift from “the LB went up” to “the gain survived untouched labels.”

## What would change my mind?

- a fixed method that improves multiple genuinely independent OOF schemes;
- a feature or external-data mechanism identified before submitting, with stable subgroup gains;
- the same adjustment transferring across unrelated strong anchors without tuning its sign or weight on LB;
- final private results.

Until then, `0.97115 > 0.97110` is an observation about one visible sample, not evidence of better generalization.

## The 0.97115 submission — and why it is probably overfit

For reproducibility, this writes the exact simple construction behind submission **55584395**:

1. take the public Naji v19 prediction;
2. rank both it and the seven-variable student;
3. use `1.08 × teacher_rank − 0.08 × student_rank`;
4. rank again, with deterministic tie-breaking.

This file is **not one of my private-leaderboard selections**. It is included because a public notebook about leaderboard overfitting should be willing to diagnose its own high score. The diagnosis is: strong base model, public-selected correction, negative honest OOF delta, high overfitting risk.

### 【解説】セル11: 0.97115 の提出を再現し、その上で「選ぶな」と言う

**何をしているか**: teacher（Naji v19）とstudentの両方をパーセンタイル順位に直し、
`1.08 × teacher_rank − 0.08 × student_rank` を計算して、決定論的タイブレークで再度順位化し提出します。

**なぜそうするのか**:
- **なぜ確率でなく順位で混ぜるのか**: 指標がAUCなので順位さえ合っていればよく、
  順位化すればスケールの違う2つのモデルを対等に混ぜられます（**ランクブレンド**）。
- **重みが `-0.08`（マイナス）である意味**: これは検証1で「OOFと正反対」と示された方向そのものです。
  つまりこの提出は、**著者が自分で「過学習の疑いが濃い」と診断した方向を、あえて再現している**ものです。
- **3つの `assert`**: idの一意性、予測値の重複ゼロ、有限値であること。
  提出前に自動で止まる**フェイルクローズドな検証**です。
  特に「予測値が全部ユニーク」を要求しているのは、strict_rankが正しく効いたことの確認になります。
- **最後のprintが結論**: 「public LB 0.97115 は既知。だが**公開スコアだけを根拠にこのファイルを選ぶな**」。

**この解説付き写しから持ち帰るべきこと**:
1. 改善を主張するときは「**選択に使っていないデータ**でも同じ方向か」を必ず確かめる。
2. **候補を多く試すほど、見かけの最高スコアは実力より上振れする**（多重検定）。
3. 全体の相関が高くても、**先端の順位はノイズで決まっている**ことがある。
4. 自分の高スコアを自分で疑えることが、最も価値のあるスキルの一つ。


In [ ]:
teacher_test = pct_rank(naji_test[TARGET].to_numpy())
student_test = pct_rank(signals["test_soft_student"])
public_weight = -0.08
raw = (1-public_weight)*teacher_test + public_weight*student_test
final_prediction = strict_rank(raw, teacher_test)

submission = pd.DataFrame({
    "id": naji_test["id"].to_numpy(),
    TARGET: final_prediction,
})
assert submission.id.is_unique
assert submission[TARGET].nunique() == len(submission)
assert np.isfinite(submission[TARGET]).all()
submission.to_csv("submission.csv", index=False)

print("wrote submission.csv", submission.shape)
print("known public LB: 0.97115 (submission 55584395)")
print("private recommendation: do not select this file on public score alone")
display(submission.head())